In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

In [2]:
file_path = '../data/processed/master_dataset.csv'

master_df = pd.read_csv(file_path)

master_df.head()

,country,year,ev_sales_share,ev_stock,ev_stock_share,charging_points
0,Norway,2024,92.0,960230.0,32.0,31000.0
1,Norway,2023,90.0,900290.0,30.0,28000.0
2,Norway,2022,89.0,790260.0,26.0,25000.0
3,Norway,2021,86.0,630230.0,22.0,19700.0
4,Norway,2020,75.0,490200.0,17.0,17300.0


In [3]:
country_missing = master_df.groupby('country').agg({
    'ev_stock': lambda x: x.isna().sum(),
    'ev_stock_share': lambda x: x.isna().sum(),
    'charging_points': lambda x: x.isna().sum(),
    'year': 'count'
}).sort_values('ev_stock', ascending=False)

country_missing.head(20)

,ev_stock,ev_stock_share,charging_points,year
country,,,,
Hungary,10,10,10,10
Czech Republic,10,10,10,10
Estonia,10,10,10,10
Romania,10,10,10,10
Slovakia,10,10,10,10
Latvia,10,10,10,10
Ireland,10,10,10,10
Bulgaria,9,9,9,9
Lithuania,9,9,9,9


In [4]:
countries_to_remove = [
    'Hungary','Czech Republic','Estonia','Romania',
    'Slovakia','Latvia','Ireland','Bulgaria',
    'Lithuania','Slovenia','Seychelles'
]

master_df = master_df[
    ~master_df['country'].isin(countries_to_remove)
]

Countries with no available observations for the selected explanatory variables (EV stock, EV stock share and charging infrastructure) were excluded from the analysis. Since these variables were entirely absent in the original IEA dataset, reliable imputation was not possible. The remaining missing observations were imputed using country-specific linear interpolation over time, followed by forward and backward filling where necessary.

In [5]:
master_df.isnull().sum()

country              0
year                 0
ev_sales_share       0
ev_stock            34
ev_stock_share      34
charging_points    185
dtype: int64

In [6]:
master_df = master_df.sort_values(['country','year'])

for col in ['ev_stock','ev_stock_share','charging_points']:
    master_df[col] = (
        master_df.groupby('country')[col]
        .transform(
            lambda x: x.interpolate()
                       .ffill()
                       .bfill()
        )
    )

In [7]:
master_df.isnull().sum()

country             0
year                0
ev_sales_share      0
ev_stock           18
ev_stock_share     18
charging_points    34
dtype: int64

In [8]:
missing_countries = master_df[
    master_df[['ev_stock','ev_stock_share','charging_points']]
    .isna()
    .any(axis=1)
]

missing_countries['country'].unique()

<StringArray>
['Croatia', 'Cyprus', 'Luxembourg', 'Middle East and Caspian', 'Uzbekistan']
Length: 5, dtype: str

In [9]:
master_df = master_df[
    master_df['country'] != 'Middle East and Caspian'
]

In [10]:
master_df['country'].nunique()

43

In [11]:
master_df = master_df.sort_values(['country','year'])

for col in ['ev_stock','ev_stock_share','charging_points']:
    master_df[col] = (
        master_df.groupby('country')[col]
        .transform(
            lambda x: x.interpolate(method='linear')
        )
    )

In [12]:
master_df.isnull().sum()

country             0
year                0
ev_sales_share      0
ev_stock           18
ev_stock_share     18
charging_points    20
dtype: int64

In [14]:
additional_remove = [
    'Croatia',
    'Cyprus',
    'Luxembourg',
    'Middle East and Caspian'
]

master_df = master_df[
    ~master_df['country'].isin(additional_remove)
]

In [15]:
master_df[master_df['charging_points'].isna()]

,country,year,ev_sales_share,ev_stock,ev_stock_share,charging_points
269,Uzbekistan,2023,2.2,9960.0,0.34,NaN
181,Uzbekistan,2024,5.0,32400.0,0.87,NaN


Countries with missing explanatory-variable data were excluded. For the remaining dataset, country-level interpolation was used where appropriate. Two observations from Uzbekistan remained missing for charging infrastructure and were removed due to insufficient information for reliable imputation.

In [16]:
master_df = master_df.dropna(subset=['charging_points'])

In [17]:
master_df.isnull().sum()

country            0
year               0
ev_sales_share     0
ev_stock           0
ev_stock_share     0
charging_points    0
dtype: int64

In [18]:
master_df.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)